# 1. RAG 평가 개요
- RAG 평가란 RAG 시스템이 주어진 입력에 대해 얼마나 효과적으로 관련 정보를 검색하고, 이를 기반으로 정확하고 유의미한 응답을 생성하는지를 측정하는 과정이다. 
- **평가 요소**
    - **검색 단계 평가**
        - 입력 질문에 대해 검색된 문서나 정보의 관련성과 정확성을 평가.
    - **생성 단계 평가**
        - 검색된 정보를 기반으로 생성된 응답의 품질, 정확성등을 평가.
- **평가 방법**
    - **온/오프라인 평가**
        1. **오프라인 평가**
            - 미리 준비된 데이터셋을 활용하여 RAG 시스템의 성능을 측정한다.
        2. **온라인 평가**
            - 실제 사용자 트래픽과 피드백을 기반으로 시스템의 실시간 성능을 평가한다.
    - **정량적/정성적 평가**
        1. 정량적 평가
            - 자동화된 지표를 사용하여 생성된 텍스트의 품질을 평가한다.
        2. 정성적 평가
            - 전문가나 일반 사용자가 직접 생성된 응답의 품질을 평가하여 주관적인 지표를 평가한다.

# 2. [RAGAS](https://www.ragas.io/)
- RAGAS는 RAG 파이프라인을 **정량적으로 평가하는** 오픈소스 프레임 워크이다. 
- RAGAS 문서: https://docs.ragas.io/en/stable/
## 2.1 설치
- `pip install ragas rapidfuzz`

## 2.2 RAGAS 평가 지표 개요
![ragas_score](figures/ragas_score.png)
- **Generation**
    - llm 모델이 생성한 답변에 대한 평가 지표들.
    - **Faithfulness(신뢰성)**
        -  생성된 답변과 검색된 문서(context)간의 관련성을 평가하는 지표
        -  생성된 답변이 주어진 문맥(context)에 얼마나 충실한지를 평가하는 지표로 할루시네이션에 대한 평가로 볼 수있다.
    - **Answer relevancy(답변 적합성)**
        - 생성된 답변과 사용자의 질문간의 관련성을 평가하는 지표
        - 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지를 평가하는 지표.
- **Retrieval**
    -  질문에 대해 검색한 문서(context)들에 대한 평가
    -  **Context Precision(문맥 정밀도)**
        -  검색된 문서(context)들 중 질문과 관련 있는 것들이 **얼마나 상위 순위에 위치하는지** 평가하는 지표.
    -  **Context Recall(문맥 재현률)**
        -  검색된 문서(context)가 정답(ground-truth)의 정보를 얼마나 포함하고 있는지 평가하는 지표.
- 이러한 지표들은 RAG 파이프라인의 성능을 다각도로 평가하는 데 활용된다.
![RAGAS_score2](figures/RAGAS_score2.png)

## 2.3 주요 평가지표
### 2.3.1 Generation 평가
- LLM이 생성한 답변에 대한 평가
  
#### 2.3.1.1 Faithfulness (신뢰성)
- 생성된 답변이 얼마나 주어진 검색 문서들(context)를 잘 반영해서 생성되었는지 평가한다. 할루시네이션에 대한 평가라고 할 수 있다. 
- 점수범위: **0 ~ 1** (1에 가까울수록 좋음)
- 답변에 포함된 모든 주장이 context에서 얼마나 추출 가능한지를 확인한다.

##### 2.3.1.1.1평가 방법
1. Answer에서 주장 구문(claim statement)들을 생성(추출)한다. (주장이란, 질문(user input)과 관련된 내용)
    - 예) 
        - **질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요? 
        - **LLM 답변**: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 3000만명이다.
2. 각 주장들을 context로 부터 추론 가능한지 판단한다. 이를 바탕으로 faithfulness 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다. .... 한국의 인구는 5000만명이고 서울에 1000만이 살고 있다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 추론가능한 주장.
            - 한국의 인구는 3000만명이다. -> context에서 추론 불가능한 주장.
3. **Faithfulness score** 를 계산한다. 총 주장 수 중에서 context로 부터 추론가능한 주장의 개수.    
    - 예)
        - Faithfulness Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 유추할 수있다.)
    - LLM 답변에서 주장을 추출 하는 것과 각 주장이 context에서 추론 가능한 지를 판단하는 것은 LLM 을 활용한다.
- 공식
    $$
    \text{Faithfulness Score}\;=\;\cfrac{\text{주어진\;context\;에서\;추론할\;수\;있는\;주장의\;개수}}{\text{총\;주장\;개수}}
    $$

### 2.3.2 Answer relevancy (답변 적합성)
- 생성된 답변이 질문(user input)에 얼마나 잘 부합하는 지를 평가한다.
- 점수 범위: -1~1 (1에 가까울수록 좋음)
- LLM이 생성한 답변을 기반으로 질문들을 생성한다. 이렇게 생성한 질문들과 실제 질문(user input) 간의 유사도를 측정한다.

#### 2.3.2.1 평가방법
1. LLM이 생성한 답변을 기반으로 질문들을 생성한다.
    - 예) 
        - **LLM** 답변: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
2. 실제 질문과 생성한 질문간의 코사인 유사도를 측정한다. 그 평균이 최종 점수가 된다.
    - 예)
        - **실제 질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요?
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
- 공식
  $$
    \cfrac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(q_{\text{user}_{_i}}, q_{\text{generated}})
  $$

## 2.4 Retrieval 평가
Vector store에서 검색한 context에 대한 평가

### 2.4.1 Context Precision
- 검색된 문서(context)들 중 질문과 관련 있는 것들이 얼마나 **상위 순위**에 있는 지 평가.
- 점수 범위: 0~1 (1에 가까울수록 좋음)


#### 2.4.1.1평가방법

- 공식
$$
 \text{Context\;Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\ 상위\;K개\;결과에서의\;관련\;항목\;수}
$$
$$
 \text{Precision@k} = \frac{\text{True\;positive@k}}{(\text{True\;positive@k} + \text{False\;positive@k})} \\
$$
- $\text{Precision@k}$: 개별 문서에 대한 Precision
- K: context 의 개수(chuck 수)
- $v_k$: 관련성 여부로 0 또는 1. (0: 관련 없음, 1: 관련 있음)

#### 2.4.1.2 예시
- 질문과 context 관련성의 예
    - 질문: 한국의 수도는 어디이고 인구는 얼마나 되나요?
    - **높은 정밀도 context들**: 질문과 직접적인 관련이 있는 문서들
        - 한국의 수도는 서울이고 인구는 5000만명 입니다. 
        - 한국의 수도는 서울입니다.
        - 한국은 동아시아에 위치해 있는 국가로 수도는 서울입니다.
        - 한국의 인구는 5000만명 입니다.
    - **낮은 정밀도 context**: 한국과 관련있어 검색될 수 있지만 질문과 직접적 관련이 없다. 
        - 한국은 동아시아에 위치한 국가입니다.
        - 한국의 K-pop은 전 세계적으로 유명합니다.
        - 비빔밥, 불고기는 한국의 대표적인 음식입니다.
    - **높은 정밀도의 context이 상위 순위에 위치했으면 높은 점수를 받는다.**

- 점수 계산 예:
    - **상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정하자.**
    - **Precision@K 계산**
        ```bash
            Precision@1 = 1/1 = 1.0    # True positive@1/(True positive@1 + False positive@1).  1/1(1번 문서 계산 시에는 1개 문서만 있으므로 분모가 1이 된다.)
            Precision@2 = 1/2 = 0.5
            Precision@3 = 2/3 ≈ 0.67    
            Precision@4 = 3/4 = 0.75
            Precision@5 = 3/5 = 0.6
        ```
    - **vk의 값**
        - 1번째: $v_1 = 1$ - 관련있음
        - 2번째: $v_2 = 0$ - 관련없음
        - 3번째: $v_3 = 1$ - 관련있음
        - 4번째: $v_4 = 1$ - 관련있음
        - 5번째: $v_5 = 0$ - 관련없음

    - **Context Precision@5**
        $$
        \text{Context\;Precision@5} = \frac{(1.0 \times 1) + (0.5 \times 0) + (0.67 \times 1) + (0.75 \times 1) + (0.6 \times 0)}{3} = \frac{1.0 + 0 + 0.67 + 0.75 + 0}{3} ≈ 0.807
        $$

### 2.4.2 Context Recall (문맥 재현률)
- 검색된 문서(context)가 얼마나 정답(ground-truth)의 정보를 포함있는 지 평가하는 지표
- 점수 범위: 0~1 (1에 가까울수록 좋음)
- **정답(ground truth)의 각 주장(claim)이 검색된 context와 얼마나 일치**하는지 계산함.

#### 2.4.2.1 평가방법
1. 정답에서 주장(claim)들을 생성(추출)한다.
    - 예) 
        - **정답**: 한국의 수도는 서울이고 인구수는 5000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 5000만명이다.
2. 각 주장(claim)의 정보를 검색된 contexts에서 찾을 수 있는지 판별한다. 이를 바탕으로 context recall 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 찾을 수 있다.
            - 한국의 인구는 5000만명이다. -> context에서 찾을 수 없다.
3. **Context Recall Score** 를 계산한다. 총 주장 수 중에서 context로 부터 찾을 수 있는 주장의 개수.
    - 예)
        - Context Recall Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 찾을 수 있다.)

- 공식
    $$
    \text{Context Recall Score}\;=\;\cfrac{\text{GT의\;주장\;중\;주어진\;context\;에서\;찾을\;수\;있는\;주장의\;개수}}{\text{GT의\;총\;주장\;개수}}
    $$ 

# 3. RAGAS 평가 실습

In [21]:
# !uv pip install ragas rapidfuzz
# 설치 후 커널 재시작

In [4]:
# docker run -p 6333:6333 -p 6334:6334   -v qdrant_storage:/qdrant/storage   qdrant/qdrant
# termianal에서 실행

In [5]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, SparseVectorParams, VectorParams
from langchain_openai import OpenAIEmbeddings

from dotenv import load_dotenv

load_dotenv()


True

## 0. 패키지 설치


In [12]:
# 필요한 library: pymupdf pdfplumber pillow openai qdrant-client langchain-core langchain-openai langchain-qdrant langchain-text-splitters python-dotenv


## 1. 설정 및 클라이언트 준비


In [13]:
import os
import re
import glob
import uuid
from pathlib import Path

import fitz  # PyMuPDF
import pdfplumber
from PIL import Image
from openai import OpenAI
from dotenv import load_dotenv

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, Filter, FieldCondition, MatchValue

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

# ----- 설정값 -----
GUIDANCE_DIR = "./길라잡이"          # 이 폴더 안의 모든 PDF를 찾아서 적재

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY") or None
COLLECTION_NAME = "guidance_vectordb"

CHAT_MODEL = "gpt-5.4-mini"                          # 표 요약 / 이미지 캡션용
EMBED_MODEL = "text-embedding-3-large"         # large 임베딩 모델
EMBED_DIM = 3072

CHUNK_SIZE = 800       # 텍스트 청크 최대 글자 수 (splitter 기준)
CHUNK_OVERLAP = 150    # 청크 간 겹치는 글자 수 (문맥 끊김 방지)

openai_client = OpenAI()
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)


def ensure_collection():
    existing = [c.name for c in qdrant_client.get_collections().collections]
    if COLLECTION_NAME in existing:
        print(f"'{COLLECTION_NAME}' collection이 이미 존재합니다.")
        return
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    )
    print(f"'{COLLECTION_NAME}' collection 생성 완료 (dim={EMBED_DIM})")


ensure_collection()

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

pdf_paths = sorted(glob.glob(str(Path(GUIDANCE_DIR) / "*.pdf")))
print(f"'{GUIDANCE_DIR}' 안에서 발견된 PDF {len(pdf_paths)}개")
for p in pdf_paths:
    print(" -", p)


'guidance_vectordb' collection이 이미 존재합니다.
'./길라잡이' 안에서 발견된 PDF 2개
 - 길라잡이\사병_길라잡이.pdf
 - 길라잡이\초급간부_길라잡이.pdf



## 2. 텍스트 정제


In [14]:
def clean_text(text: str) -> str:
    # Private Use Area (U+E000–U+F8FF) 문자 제거
    text = re.sub(r'[\ue000-\uf8ff]', '', text)
    # 과도한 공백/빈 줄 정리
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()



## 3. 표 추출 → 마크다운 → 요약 


In [16]:
def table_to_markdown(table_rows):
    rows = [[cell if cell is not None else "" for cell in row] for row in table_rows]
    if not rows:
        return ""
    header = rows[0]
    body = rows[1:]
    md_lines = ["| " + " | ".join(header) + " |", "| " + " | ".join(["---"] * len(header)) + " |"]
    for row in body:
        md_lines.append("| " + " | ".join(row) + " |")
    return "\n".join(md_lines)


def extract_tables(pdf_path: str):
    tables_out = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            for table in page.extract_tables():
                if not table or len(table) < 2:
                    continue
                md_table = table_to_markdown(table)
                tables_out.append({"page": page_num, "markdown": md_table})
    return tables_out


def summarize_table(table_markdown: str) -> str:
    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        max_completion_tokens=300,
        messages=[{
            "role": "user",
            "content": (
                "다음은 문서에서 추출한 표입니다. 이 표가 어떤 내용을 담고 있는지 "
                "검색에 활용할 수 있도록 2~4문장의 한국어로 요약해줘. "
                "컬럼이 의미하는 바와 주요 수치/항목을 포함해줘.\n\n"
                f"{table_markdown}"
            ),
        }],
    )
    return response.choices[0].message.content.strip()



## 4. 텍스트 청킹 (페이지 단위)

이 PDF는 법령처럼 문장이 페이지를 넘어 쭉 이어지는 문서가 아니라, 페이지 하나하나가 그 자체로 하나의 토픽/안내 단위로 디자인된 문서입니다. 그래서 페이지 경계를 문맥 단위로 그대로 씁니다: **원칙적으로 한 페이지 = 한 청크**입니다.


In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

# 이 길이를 넘는 페이지만 splitter로 추가 분할 (그 외엔 페이지 통째로 한 청크)
PAGE_OVERFLOW_THRESHOLD = 1200


def chunk_text_by_page(pdf_path: str) -> list[dict]:
    """페이지 단위로 청크를 만든다. 페이지가 너무 길 때만 그 페이지 안에서 추가로 나눈다."""
    doc = fitz.open(pdf_path)
    results = []

    for page_num, page in enumerate(doc, start=1):
        page_text = clean_text(page.get_text("text"))

        if not page_text:
            continue  # 텍스트 없는 페이지(이미지만 있는 페이지 등)는 건너뜀

        if len(page_text) <= PAGE_OVERFLOW_THRESHOLD:
            results.append({"page": page_num, "text": page_text})
        else:
            # 이 페이지만 너무 길어서 임베딩이 흐려질 수 있는 경우 -> 페이지 내부에서만 분할
            sub_chunks = text_splitter.split_text(page_text)
            for sub in sub_chunks:
                results.append({"page": page_num, "text": sub})

    doc.close()
    return results



## 6. PDF 1개 → LangChain `Document` 리스트로 통합


In [18]:
ID_NAMESPACE = uuid.NAMESPACE_URL


def make_id(doc_name: str, chunk_type: str, index: int) -> str:
    """같은 PDF를 다시 적재해도 같은 ID가 나오도록 결정론적으로 생성 (재실행 시 upsert로 덮어씀)"""
    return str(uuid.uuid5(ID_NAMESPACE, f"{doc_name}::{chunk_type}::{index}"))


def build_documents_for_pdf(pdf_path: str) -> tuple[list[Document], list[str]]:
    doc_name = Path(pdf_path).name
    documents = []
    ids = []
    counter = 0

    # 1) 텍스트 청크
    text_chunks = chunk_text_by_page(pdf_path)
    for c in text_chunks:
        documents.append(Document(
            page_content=c["text"],
            metadata={"doc_name": doc_name, "page": c["page"], "type": "text"},
        ))
        ids.append(make_id(doc_name, "text", counter))
        counter += 1

    # 2) 표
    tables = extract_tables(pdf_path)
    for t in tables:
        summary = summarize_table(t["markdown"])
        documents.append(Document(
            page_content=summary,
            metadata={
                "doc_name": doc_name, "page": t["page"], "type": "table",
                "table_markdown": t["markdown"], "summary": summary,
            },
        ))
        ids.append(make_id(doc_name, "table", counter))
        counter += 1

    print(f"[{doc_name}] 텍스트 {len(text_chunks)} / 표 {len(tables)} → 총 {len(documents)}개")
    return documents, ids



## 7. 디렉토리 전체 적재 (upsert)


In [19]:

def upsert_documents(documents: list[Document], ids: list[str], batch_size: int = 100):
    for i in range(0, len(documents), batch_size):
        batch_docs = documents[i:i + batch_size]
        batch_ids = ids[i:i + batch_size]
        vectorstore.add_documents(documents=batch_docs, ids=batch_ids)
        print(f"  upsert 진행: {min(i + batch_size, len(documents))}/{len(documents)}")


def run_pipeline(directory: str = GUIDANCE_DIR):
    paths = sorted(glob.glob(str(Path(directory) / "*.pdf")))
    print(f"'{directory}'에서 PDF {len(paths)}개 발견, 적재 시작\n")

    total_chunks = 0
    for pdf_path in paths:
        documents, ids = build_documents_for_pdf(pdf_path)
        upsert_documents(documents, ids)
        total_chunks += len(documents)
        print()

    print(f"완료: PDF {len(paths)}개 / 총 {total_chunks}개 청크 적재")


# 실행
run_pipeline(GUIDANCE_DIR)


'./길라잡이'에서 PDF 2개 발견, 적재 시작

[사병_길라잡이.pdf] 텍스트 84 / 표 59 → 총 143개
  upsert 진행: 100/143
  upsert 진행: 143/143

[초급간부_길라잡이.pdf] 텍스트 92 / 표 102 → 총 194개
  upsert 진행: 100/194
  upsert 진행: 194/194

완료: PDF 2개 / 총 337개 청크 적재



## 8. 검색 예시
`vectorstore.similarity_search`로 기본 검색, `filter`로 `type`(text/table/image) 등 메타데이터 필터링도 가능합니다.

In [22]:
def search(query: str, k: int = 5, type_filter: str = None):
    qfilter = None
    if type_filter:
        qfilter = Filter(must=[FieldCondition(key="metadata.type", match=MatchValue(value=type_filter))])

    results = vectorstore.similarity_search_with_score(query, k=k, filter=qfilter)

    for doc, score in results:
        print(f"[score={score:.3f}] doc={doc.metadata.get('doc_name')} page={doc.metadata.get('page')} type={doc.metadata.get('type')}")
        print(doc.page_content[:200])
        print("-" * 40)

    return results


# 사용 예시:
search("군인 해택 휴양시설알려줘")
# search("휴가 관련 표", type_filter="table")
# search("기차 할인 안내 이미지", type_filter="image")



[score=0.585] doc=사병_길라잡이.pdf page=23 type=text
2026 병 복지 길라잡이 23
 • 국군복지단 휴양시설 현황
구분
시 설 명
위 치
전화번호(군)
전화번호(일반)
직영
서귀포 호텔
제주 서귀포시
960-7703
064-738-0123
화진포 콘도
강원 고성군 
960-7706
033-682-0500
청간정 콘도
강원 고성군 
822-6470
033-631-9331
송 정 콘도
강원 강릉시 
940-38
----------------------------------------
[score=0.546] doc=사병_길라잡이.pdf page=22 type=text
Part 01 생활편의 분야
22 2026 병 복지 길라잡이
휴양시설 이용
이용대상 / 방법 및 혜택
 • 이용대상 : 현역병, 배우자 및 그 직계 존·비속
 * 가족의 경우 가족관계 증빙서류, 신분증 또는 국방가족 모바일증명 제시
 * 국방가족 모바일증명은 국군복지포털 홈페이지를 이용하여 등록
 • 국군복지단 휴양시설 이용방법
 - 국군복지포털 홈페이지 
----------------------------------------
[score=0.545] doc=초급간부_길라잡이.pdf page=35 type=text
34 
2026 초급간부 길라잡이
•국군복지단 휴양시설 현황
구분
시 설 명
위 치
전화번호(군)
전화번호(일반)
직영
서귀포 호텔
제주 서귀포시
960-7703
064-738-0123
화진포 콘도
강원 고성군 
960-7706
033-682-0500
청간정 콘도
강원 고성군 
988-7873
033-631-9331
송정 콘도
강원 강릉시 
940-3881
----------------------------------------
[score=0.528] doc=초급간부_길라잡이.pdf page=34 type=text
02
Part
생
활
편
의
 
분
야
33 
2026 초급간부 길라잡이
•국군복지단 초급간부전용 객실예약 이용방법
- 국군복지포털 호텔/

[(Document(metadata={'doc_name': '사병_길라잡이.pdf', 'page': 23, 'type': 'text', '_id': 'fb4aa98d-c931-5260-82b6-e1fa8da946b5', '_collection_name': 'guidance_vectordb'}, page_content='2026 병 복지 길라잡이 23\n • 국군복지단 휴양시설 현황\n구분\n시 설 명\n위 치\n전화번호(군)\n전화번호(일반)\n직영\n서귀포 호텔\n제주 서귀포시\n960-7703\n064-738-0123\n화진포 콘도\n강원 고성군 \n960-7706\n033-682-0500\n청간정 콘도\n강원 고성군 \n822-6470\n033-631-9331\n송 정 콘도\n강원 강릉시 \n940-3881\n033-652-7573\n대 천 콘도\n충남 보령시 \n960-7705\n041-932-6305\n민영\n한 화\n설악, 용인, 대천, 산정호수, \n해운대, 경주, 제주, 평창, 거제\n국군복지단\n예약실\n984-6584\n국군복지단\n예약실\n1577-9800\n내선 “1번”\n금 호\n통영, 화순, 설악, 제주, 아산\n소 노\n설악, 경주, 단양, 홍천, 양양, \n변산, 거제, 삼척, 진도, 여수, \n고양(호텔), 양평\n켄싱턴\n남원, 설악비치, 충주, 경주,\n가평, 설악밸리, 하동, 서귀포\n무 주\n전북무주\n리 솜\n안면도\n엘도라도\n전남신안\n샤인빌(소노캄)\n제주\n보 광\n제주\n웰리힐리\n강원횡성\n영랑호\n강원 속초\n현대수\n강원 속초\n • 각 군 휴양시설 이용방법 : 각 군 홈페이지 참조\n구분\n시 설 명\n홈페이지 주소\n국방부\n국방컨벤션(웨딩)\nwww.mndconvention.co.kr\n육군\n로카우스 호텔, 계룡스파텔\nwelfare.army.mil.kr\n해군\n해군호텔(서울, 평택, 제주), \n해군회관(진해)\nwelfare.navy.mil.kr\n공군\n공군호텔\nwww.airforcehote

In [48]:
# ! uv pip install nbconvert

In [ ]:
# ! jupyter nbconvert --to script guidance_loader.ipynb
# guidance_loader.ipynb -> guidance_loader.py

In [6]:
##################################################
# vector DB 연결
# retriever 생성
##################################################

from pathlib import Path
import glob

from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

from guidance_loader import build_documents_for_pdf, GUIDANCE_DIR  # 실제 존재하는 함수/상수로 교체

EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM = 3072


def get_vectorstore(collection_name: str = "guidance_vectordb", force_rebuild: bool = False):
    """
    벡터스토어에 연결한다.
    - 컬렉션이 이미 있으면: 연결만 하고 재사용 (재적재로 인한 시간/비용 낭비 방지)
    - 컬렉션이 없으면: 새로 만들고 PDF들을 적재
    - force_rebuild=True: 기존 컬렉션을 지우고 처음부터 다시 적재
    """
    dense_embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
    client = QdrantClient(url="http://localhost:6333")

    collection_exists = client.collection_exists(collection_name)

    if force_rebuild and collection_exists:
        client.delete_collection(collection_name=collection_name)
        collection_exists = False

    if not collection_exists:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
        )

    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=dense_embeddings,
    )

    # 새로 만든 컬렉션일 때만 적재 (기존 컬렉션 있으면 스킵)
    if not collection_exists:
        _ingest_documents(vectorstore)

    return vectorstore


def _ingest_documents(vectorstore: QdrantVectorStore, directory: str = GUIDANCE_DIR, batch_size: int = 100):
    """guidance_loader.ipynb의 build_documents_for_pdf + upsert 로직을 그대로 재사용"""
    paths = sorted(glob.glob(str(Path(directory) / "*.pdf")))
    print(f"'{directory}'에서 PDF {len(paths)}개 발견, 적재 시작\n")

    total_chunks = 0
    for pdf_path in paths:
        documents, ids = build_documents_for_pdf(pdf_path)
        for i in range(0, len(documents), batch_size):
            vectorstore.add_documents(
                documents=documents[i:i + batch_size],
                ids=ids[i:i + batch_size],
            )
        total_chunks += len(documents)
        print()

    print(f"완료: PDF {len(paths)}개 / 총 {total_chunks}개 청크 적재")


def get_retriever(vectorstore, k: int = 5):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever

'guidance_vectordb' collection이 이미 존재합니다.
'./길라잡이' 안에서 발견된 PDF 2개
 - 길라잡이\사병_길라잡이.pdf
 - 길라잡이\초급간부_길라잡이.pdf
'./길라잡이'에서 PDF 2개 발견, 적재 시작

[사병_길라잡이.pdf] 텍스트 84 / 표 59 → 총 143개
  upsert 진행: 100/143
  upsert 진행: 143/143

[초급간부_길라잡이.pdf] 텍스트 92 / 표 102 → 총 194개
  upsert 진행: 100/194
  upsert 진행: 194/194

완료: PDF 2개 / 총 337개 청크 적재


In [7]:
vectorstore = get_vectorstore()
print(vectorstore)
retriever = get_retriever(vectorstore)
print(retriever)

tags=['QdrantVectorStore', 'OpenAIEmbeddings'] vectorstore=<langchain_qdrant.qdrant.QdrantVectorStore object at 0x000001CFF4193890> search_kwargs={'k': 5}


In [8]:
################################################################################
# 평가할 RAG Chain
################################################################################

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI

vectorstore = get_vectorstore()
retriever = get_retriever(vectorstore)

prompt_txt = """<instruction>
당신은 정보제공을 목적으로하는 유능한 AI Assistant 입니다.
주어진 context의 내용을 기반으로 질문에 답변을 합니다.
Context에 질문에 대한 명확한 정보가 있는 경우 그것을 바탕으로 답변을 합니다.
Context에 질문에 대한 명확한 정보가 없는 경우 "정보가 부족해 답을 할 수없습니다." 라고 답합니다.
절대 추측이나 일반 상식을 바탕으로 답을 하거나 Context 없는 내용을 만들어서 답변해서는 안됩니다.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
"""
prompt = ChatPromptTemplate.from_template(template=prompt_txt)

model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()


def format_docs_for_prompt(documents: list[Document]) -> str:
    """LLM 프롬프트({context})용: 문서들을 하나의 문자열로 합침 (doc_name/page도 같이 표기)"""
    parts = []
    for doc in documents:
        meta = doc.metadata
        header = f"[{meta.get('doc_name', '')} p.{meta.get('page', '')}]"
        parts.append(f"{header}\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


def format_docs_for_ragas(documents: list[Document]) -> list[str]:
    """RAGAS 평가용: 검색된 문서 각각의 내용을 list[str]로 추출"""
    return [doc.page_content for doc in documents]


def build_prompt_input(state: dict) -> dict:
    """검색 결과({'documents':..., 'query':...})를 prompt 입력({'context':str, 'query':str})으로 변환"""
    return {
        "context": format_docs_for_prompt(state["documents"]),
        "query": state["query"],
    }


# 1단계: 질문으로 검색 → 원본 Document 리스트 + 질문 보존 (retriever는 여기서 1회만 호출됨)
retrieve_step = {
    "documents": retriever,
    "query": RunnablePassthrough(),
}

# RAG 체인 -> RAG 평가 데이터셋을 만드는 RAG Chain -> 최종응답: LLM 응답(str), 검색한 문서들(list[str]), 질문(str)
chain = (
    RunnablePassthrough()  # dict | dict 가 파이썬 dict 병합 연산으로 취급되는 것을 막기 위해 필요
    | retrieve_step
    | {
        "response": RunnableLambda(build_prompt_input) | prompt | model | parser,
        "retrieved_context": RunnableLambda(lambda state: format_docs_for_ragas(state["documents"])),
        "query": RunnableLambda(lambda state: state["query"]),
    }
)

In [9]:
res = chain.invoke("병장의 월급은?")

In [10]:
res["response"]

'병장의 월급은 **1,500,000원**입니다.'

In [11]:
res["retrieved_context"]

['Part 01 생활편의 분야\n6 2026 병 복지 길라잡이\n보 수\n봉 급\n • 계급별 지급액\n 단위 : 원\n \n이병\n일병\n상병\n병장\n750,000\n900,000\n1,200,000\n1,500,000\n • 병 봉급 인상\n 단위 : 원\n연도\n계급\n’21년\n’22년\n’23년\n’24년\n’25~’26년\n병 장 \n608,500\n676,100\n1,000,000\n1,250,000\n1,500,000\n상 병 \n549,200\n610,200\n800,000\n1,000,000\n1,200,000\n일 병\n496,900\n552,100\n680,000\n800,000\n900,000\n이 병\n459,100\n510,100\n600,000\n640,000\n750,000\n인상율(병장)\n12.5%\n11.1%\n47.9%\n25%\n20%\n* 병역의무 이행자의 보상과 예우를 위해 병 봉급과 자산형성프로그램을 결합하여\n’25년까지 병장 기준 200만원 이상으로 인상\n수 당\n단위 : 원\n \n구 분\n지급 기준\n금액\n특수지\n근 무\n수 당\n갑\n・비무장지대, 서해5도, 울릉도, 접적 해역 상주 근무자\n * 가산금 : 서해5도 40,000원, 비무장지대 및 \n북방한계선 인접해역 20,000원\n25,000 \n(가산금별도)\n을\n・GOP, 해안초소, 해발 800m 이상 고지대 상주 근무자\n * 가산금 : 10,000원(GOP / 해안초소)\n20,000\n(가산금별도)\n항 공\n수 당\n을\n・항공기 동승근무자\n40,000 \n함 정\n수 당\n을\n・전투함 및 지원함 승무원\n * 함정출동가산금 : 1일 4,000원\n32,700 \n(가산금별도)\n병\n・수륙양용궤도차량 승무원\n32,700 \npart 01\n생활편의 분야\n▶▶▶ 병 복지혜택 이렇게 다양합니다',
 '이 표는 **계급별 연도(’21~’26년) 급여 수준과 병장 급여 인상률**을 비교한 자료입니다. 행은 **병장·상병·일병·이병

# 4. RAGAS 를 이용해 평가를 위한 합성 데이터 셋 만들기

- 평가 데이터셋 구성
  - `user_input`: 사용자 질문
  - `retrieved_contexts`: Vectorstore에서 검색한 context
  - `response`: LLM의 응답
  - `reference`: 정답

## 4.1 TestsetGenerator
- **문서(retrieved_contexts)를 기준**으로 **질문**, **정답** 을 생성한다.
- 평가할 LLM으로 생성된 질문을 넣어 답변을 추출하여 데이터셋을 구성한다.


> **주의**
> - TestsetGenerator import 시 `No Module named langchain_community.chat_models.vertexai` Error 발생 
> - RAGAS와 langchain-community의 버전 호환성 문제 때문에 발생한다.
> - 해결
>   1. langchain_google_vertexai 설치
>       - `!uv pip install langchain_google_vertexai`
>   2. `.venv\Lib\site-packages\langchain_community\chat_models` 디렉토리 아래 `vertexai.py` 파일을 만들고 아래 코드를 > 넣는다.
>    ```python
>       try:
>          from langchain_google_vertexai import ChatVertexAI
>       except ImportError:
>           class ChatVertexAI:
>               def __init__(self, *args, **kwargs):
>                   raise ImportError(
>                       "ChatVertexAI requires langchain-google-vertexai. "
>                       "Install with: pip install langchain-google-vertexai"
>                   )
>    ```

In [12]:
# !uv pip install langchain_google_vertexai

In [31]:
# 주피터노트북 환경에서 비동기적 처리 위해 
# script(.py) 로 작성할 경우는 필요 없다.

import nest_asyncio
nest_asyncio.apply()

In [13]:
from ragas.testset import TestsetGenerator

c:\Users\Playdata\SKN\SKN31-3rd-2Team\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
#
# testset -> Context들(문서들) - [질문 - 정답답변 + (Retriever가 찾은 문서 + LLM 응답: chain에서 생성)]
# 1. Context(문서들)을 추출 - TestsetGenerator -> 질문과 정답 답변 생성

import random

# 데이터셋을 생성할 때 사용할 Context를 추출
client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "guidance_vectordb"

info = client.get_collection(COLLECTION_NAME)
total_docs = info.points_count

results, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=total_docs,
)

# random하게 k(5)개를 sampling
sample_docs:"list[PointStruct]" =random.sample(results, 5) # 리스트에서 랜덤하게 k(5)개를 추출

# PointStruct - payload: page_content, metadata
# page_content만 추출해서 list[str]
docs = [point.payload['page_content'] for point in sample_docs]



In [15]:
total_docs

337

In [16]:
sample_docs

[Record(id='60fe2294-6088-5447-95a5-5742cc5b3031', payload={'page_content': '01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \n분\n야\n67 \n03\nPart\n2026 초급간부 길라잡이\n-③-Ⅵ\n국민내일배움카드(고용노동부) \n-③-Ⅶ\n기 타\n 개 요 \n \x07전 생애에 걸친 직무수행능력 습득·향상을 위해 국민 스스로 직업능력개발훈련을 \n실시할 수 있도록 훈련비 등 지원\n* 단, 관련 법령에 따라 공무원, 군인 등 일부 대상 제외\n* \x07군인의 경우, 국민 평생 직업능력 개발법 시행령 제6조 및 군인사법 시행령 제60조의2에 \n따라 단기복무 전역예정자의 경우 예외적으로 지원\n 주요내용\n •(신청자격) 신청일 기준 ① 5년 미만의 기간동안 현역으로 복무하고, \n ② 전역예정일까지의 기간이 6개월 이하인 장교·준사관 또는 부사관\n •(지원내용) 계좌발급일로부터 5년간 300만원 이내 훈련비 지원*\n* 훈련과정 직종별 취업률, 소득수준 등에 따라 훈련비 일정비율(0~55%) 자비 부담\n •(신청절차)\n신청인\n⇨\n각 군\n⇨\n관할 고용센터\n⇨\n훈련생·훈련기관\n• \x07신청서 작성 후 \n소속부대 담당자\n에게 제출\n•신청자격 확인 \n•\x07관할 고용센터에\n 추천 공문 발송\n•\x07신청인이 방문하여 \n발급절차 진행\n•\x07개인별 훈련과정 신청 \n및 실시\n* \x07(참고) 군 전직 및 취업 지원 업무에 관한 훈령 제11조의2 및 별지 제8호 서식(훈련과정) \n한국기술교육대학교(직업능력심사평가원) 등의 심사결과에 따라 지방관서의 장이 인정·\n통지하는 훈련과정\n* \x07국민내일배움카드 사업내용 및 훈련과정 등 구체적인 내용은 고용24 홈페이지 (www.work24.\ngo.kr)에서 확인\n 취업정보 제공 \n •국방일보 취업정보 게재 (매주 화요일) : (http://kookbang.dema.mil.k

In [17]:
docs

['01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \n분\n야\n67 \n03\nPart\n2026 초급간부 길라잡이\n-③-Ⅵ\n국민내일배움카드(고용노동부) \n-③-Ⅶ\n기 타\n 개 요 \n \x07전 생애에 걸친 직무수행능력 습득·향상을 위해 국민 스스로 직업능력개발훈련을 \n실시할 수 있도록 훈련비 등 지원\n* 단, 관련 법령에 따라 공무원, 군인 등 일부 대상 제외\n* \x07군인의 경우, 국민 평생 직업능력 개발법 시행령 제6조 및 군인사법 시행령 제60조의2에 \n따라 단기복무 전역예정자의 경우 예외적으로 지원\n 주요내용\n •(신청자격) 신청일 기준 ① 5년 미만의 기간동안 현역으로 복무하고, \n ② 전역예정일까지의 기간이 6개월 이하인 장교·준사관 또는 부사관\n •(지원내용) 계좌발급일로부터 5년간 300만원 이내 훈련비 지원*\n* 훈련과정 직종별 취업률, 소득수준 등에 따라 훈련비 일정비율(0~55%) 자비 부담\n •(신청절차)\n신청인\n⇨\n각 군\n⇨\n관할 고용센터\n⇨\n훈련생·훈련기관\n• \x07신청서 작성 후 \n소속부대 담당자\n에게 제출\n•신청자격 확인 \n•\x07관할 고용센터에\n 추천 공문 발송\n•\x07신청인이 방문하여 \n발급절차 진행\n•\x07개인별 훈련과정 신청 \n및 실시\n* \x07(참고) 군 전직 및 취업 지원 업무에 관한 훈령 제11조의2 및 별지 제8호 서식(훈련과정) \n한국기술교육대학교(직업능력심사평가원) 등의 심사결과에 따라 지방관서의 장이 인정·\n통지하는 훈련과정\n* \x07국민내일배움카드 사업내용 및 훈련과정 등 구체적인 내용은 고용24 홈페이지 (www.work24.\ngo.kr)에서 확인\n 취업정보 제공 \n •국방일보 취업정보 게재 (매주 화요일) : (http://kookbang.dema.mil.kr)\n •온라인(인터넷)을 통한 정보확인\n - \x07국방전직교육원(www.moti.or.kr) \n - 고용부 워크넷(www.wo

In [18]:
# testset 생성
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
#testsetgenerator 는 gpt-5 이후 버전은 사용할 수 없다. 
## Langchain의 모델과 Embedding 모델 -> ragas에서 사용할 수 있도록 변환(wrapping)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    llm_context="""
- 사람들이 현역병 복지에 대해서 궁금해 할 만한 질문들을 생성한다.
- 데이터셋은 반드시 한국어로 작성한다.
- 데이터셋은 JSON 문법읋 지켜서 작성한다. 특히 구두점은 꼭 지켜야 한다.
- 생성된 내용이나 Document에 JSON문법에 맞지 않는 표현이 있으면 반드시 수정해서 처리한다.
""" 
# 질문/답변을 생성할 때 LLM에게 전달할 system Prompt를 설정
)





C:\Users\Playdata\AppData\Local\Temp\ipykernel_37040\3702810314.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_37040\3702810314.py:9: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


In [19]:
testset = generator.generate_with_chunks(
    docs, testset_size=10, # Context 내용 테스트셋 개수(질문 답변 개수)
)

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]Node defe1727-0887-4e4c-891b-de296d550c7b does not have a summary. Skipping filtering.
Node 3d3e72a2-fdd4-44ed-939b-933d60534d8d does not have a summary. Skipping filtering.
Node 23435d5b-6ca7-4542-bcd5-25b785cb5878 does not have a summary. Skipping filtering.
Node 995bf862-5888-4045-96bf-01686e80fd03 does not have a summary. Skipping filtering.
Node 03c99af3-7805-42bb-a503-19a3858cd15f does not have a summary. Skipping filtering.
Applying OverlapScoreBuilder: 100%|██████████| 1/1 [00:00<00:00, 512.88it/s]
Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.
Generating Samples: 100%|██████████| 8/8 [00:02<00:00,  2.91it/s]


In [21]:
testset

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='국방일보 취업정보 어디서 볼수 있나요?', retrieved_contexts=None, reference_contexts=['01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \n분\n야\n67 \n03\nPart\n2026 초급간부 길라잡이\n-③-Ⅵ\n국민내일배움카드(고용노동부) \n-③-Ⅶ\n기 타\n 개 요 \n \x07전 생애에 걸친 직무수행능력 습득·향상을 위해 국민 스스로 직업능력개발훈련을 \n실시할 수 있도록 훈련비 등 지원\n* 단, 관련 법령에 따라 공무원, 군인 등 일부 대상 제외\n* \x07군인의 경우, 국민 평생 직업능력 개발법 시행령 제6조 및 군인사법 시행령 제60조의2에 \n따라 단기복무 전역예정자의 경우 예외적으로 지원\n 주요내용\n •(신청자격) 신청일 기준 ① 5년 미만의 기간동안 현역으로 복무하고, \n ② 전역예정일까지의 기간이 6개월 이하인 장교·준사관 또는 부사관\n •(지원내용) 계좌발급일로부터 5년간 300만원 이내 훈련비 지원*\n* 훈련과정 직종별 취업률, 소득수준 등에 따라 훈련비 일정비율(0~55%) 자비 부담\n •(신청절차)\n신청인\n⇨\n각 군\n⇨\n관할 고용센터\n⇨\n훈련생·훈련기관\n• \x07신청서 작성 후 \n소속부대 담당자\n에게 제출\n•신청자격 확인 \n•\x07관할 고용센터에\n 추천 공문 발송\n•\x07신청인이 방문하여 \n발급절차 진행\n•\x07개인별 훈련과정 신청 \n및 실시\n* \x07(참고) 군 전직 및 취업 지원 업무에 관한 훈령 제11조의2 및 별지 제8호 서식(훈련과정) \n한국기술교육대학교(직업능력심사평가원) 등의 심사결과에 따라 지방관서의 장이 인정·\n통지하는 훈련과정\n* \x07국민내일배움카드 사업내용 및 훈련과정 등 구체적인 내용은 고용24 홈페이지 (www.work24.\ngo.kr)에서 확인\

In [22]:
sample1 = testset.samples[0].eval_sample
print("사용자질문:", sample1.user_input)
print("Context:", sample1.reference_contexts)
print("생성된 답변(정답):", sample1.reference)
print("평가대상 RAG의 답변:", sample1.response)
print("평가대상 RAG가 검색한 Context:", sample1.retrieved_contexts)


사용자질문: 국방일보 취업정보 어디서 볼수 있나요?
Context: ['01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \n분\n야\n67 \n03\nPart\n2026 초급간부 길라잡이\n-③-Ⅵ\n국민내일배움카드(고용노동부) \n-③-Ⅶ\n기 타\n 개 요 \n \x07전 생애에 걸친 직무수행능력 습득·향상을 위해 국민 스스로 직업능력개발훈련을 \n실시할 수 있도록 훈련비 등 지원\n* 단, 관련 법령에 따라 공무원, 군인 등 일부 대상 제외\n* \x07군인의 경우, 국민 평생 직업능력 개발법 시행령 제6조 및 군인사법 시행령 제60조의2에 \n따라 단기복무 전역예정자의 경우 예외적으로 지원\n 주요내용\n •(신청자격) 신청일 기준 ① 5년 미만의 기간동안 현역으로 복무하고, \n ② 전역예정일까지의 기간이 6개월 이하인 장교·준사관 또는 부사관\n •(지원내용) 계좌발급일로부터 5년간 300만원 이내 훈련비 지원*\n* 훈련과정 직종별 취업률, 소득수준 등에 따라 훈련비 일정비율(0~55%) 자비 부담\n •(신청절차)\n신청인\n⇨\n각 군\n⇨\n관할 고용센터\n⇨\n훈련생·훈련기관\n• \x07신청서 작성 후 \n소속부대 담당자\n에게 제출\n•신청자격 확인 \n•\x07관할 고용센터에\n 추천 공문 발송\n•\x07신청인이 방문하여 \n발급절차 진행\n•\x07개인별 훈련과정 신청 \n및 실시\n* \x07(참고) 군 전직 및 취업 지원 업무에 관한 훈령 제11조의2 및 별지 제8호 서식(훈련과정) \n한국기술교육대학교(직업능력심사평가원) 등의 심사결과에 따라 지방관서의 장이 인정·\n통지하는 훈련과정\n* \x07국민내일배움카드 사업내용 및 훈련과정 등 구체적인 내용은 고용24 홈페이지 (www.work24.\ngo.kr)에서 확인\n 취업정보 제공 \n •국방일보 취업정보 게재 (매주 화요일) : (http://kookbang.dema.mil.kr)\n •온라인(인터넷)을 통한 정보확인\n - \x07국방전직교

In [23]:
# 생성된 Testset을 Pandas DataFrame으로 변환
eval_df = testset.to_pandas()

In [24]:
eval_df.shape

(8, 7)

In [25]:
eval_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,국방일보 취업정보 어디서 볼수 있나요?,[01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \...,국방일보 취업정보는 매주 화요일 국방일보 홈페이지(http://kookbang.de...,Job Seeker in Career Transition,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
1,국방일보에서 국민내일배움카드 관련 취업정보를 어떻게 확인할 수 있나요?,[01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \...,"국방일보에서는 매주 화요일 취업정보를 게재하며, 국민내일배움카드 관련 취업정보도 확...",Soon-to-be Discharged Service Member,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
2,장교 선발계획이 언제 어디에 올라오는지 알 수 있나요? 그리고 장교로 장기복무를 준...,[이 표는 **각 군(육군·해군·공군·해병)의 장기복무(연장복무) 선발 안내**를 ...,장교 선발계획은 각 군별로 홈페이지에 게시되는 시기가 다릅니다. 육군은 장교 전반기...,Soon-to-be Discharged Service Member,MISSPELLED,LONG,single_hop_specific_query_synthesizer
3,부사관 장기복무 선발계획은 언제 확인할 수 있습니까?,[이 표는 **각 군(육군·해군·공군·해병)의 장기복무(연장복무) 선발 안내**를 ...,"부사관 장기복무 선발계획은 육군 4월, 해군 전반기 1월과 후반기 6월, 공군 전반...",Military Career Progression Advisor,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
4,③ 취업알선 단계에서 제공되는 맞춤형 취업 지원 서비스에는 어떤 내용이 포함되어 있...,[이 표는 **취업알선 서비스의 구성 내용**을 정리한 것입니다. 주요 항목으로는 ...,"③ 취업알선 단계에서 제공되는 맞춤형 취업 지원 서비스에는 채용정보 제공, 직무 및...",Job Seeker in Career Transition,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
5,취업알선 서비스에는 어떤 지원이 포함되어 있나요?,[이 표는 **취업알선 서비스의 구성 내용**을 정리한 것입니다. 주요 항목으로는 ...,"취업알선 서비스에는 채용정보 제공, 직무/기업분석, 그룹별 취업컨설팅, 희망기업 알...",Job Seeker in Career Transition,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
6,"국군재정관리단장 뭐 하는 사람인지 궁금해요, 사망보상금 받을 때 국군재정관리단장 역...",[Part 05 재해보상 분야\n58 2026 병 복지 길라잡이\n 재해보상금의 종...,사망보상금 받을 때 국군재정관리단장은 중요한 역할을 해요. 순직여부가 각 군 전공사...,Job Seeker in Career Transition,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
7,각 군 전공사상심사위원회는 어떤 역할을 하나요?,[Part 05 재해보상 분야\n58 2026 병 복지 길라잡이\n 재해보상금의 종...,각 군 전공사상심사위원회에서는 순직여부를 결정합니다.,Military Career Progression Advisor,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer


In [26]:
from tools import search_guidance_knowledge_base
row_idx = 0
q = eval_df.loc[row_idx, 'user_input']
# resp = search_guidance_knowledge_base.invoke(q)
resp = chain.invoke(q)


In [27]:
# resp
resp['response']

'국방일보 취업정보는 매주 화요일에 국방일보에서 확인할 수 있으며, 웹사이트는 다음과 같습니다.\n\n- http://kookbang.dema.mil.kr\n\n'

In [28]:
# resp
resp['retrieved_context']

['01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \n분\n야\n67 \n03\nPart\n2026 초급간부 길라잡이\n-③-Ⅵ\n국민내일배움카드(고용노동부) \n-③-Ⅶ\n기 타\n 개 요 \n \x07전 생애에 걸친 직무수행능력 습득·향상을 위해 국민 스스로 직업능력개발훈련을 \n실시할 수 있도록 훈련비 등 지원\n* 단, 관련 법령에 따라 공무원, 군인 등 일부 대상 제외\n* \x07군인의 경우, 국민 평생 직업능력 개발법 시행령 제6조 및 군인사법 시행령 제60조의2에 \n따라 단기복무 전역예정자의 경우 예외적으로 지원\n 주요내용\n •(신청자격) 신청일 기준 ① 5년 미만의 기간동안 현역으로 복무하고, \n ② 전역예정일까지의 기간이 6개월 이하인 장교·준사관 또는 부사관\n •(지원내용) 계좌발급일로부터 5년간 300만원 이내 훈련비 지원*\n* 훈련과정 직종별 취업률, 소득수준 등에 따라 훈련비 일정비율(0~55%) 자비 부담\n •(신청절차)\n신청인\n⇨\n각 군\n⇨\n관할 고용센터\n⇨\n훈련생·훈련기관\n• \x07신청서 작성 후 \n소속부대 담당자\n에게 제출\n•신청자격 확인 \n•\x07관할 고용센터에\n 추천 공문 발송\n•\x07신청인이 방문하여 \n발급절차 진행\n•\x07개인별 훈련과정 신청 \n및 실시\n* \x07(참고) 군 전직 및 취업 지원 업무에 관한 훈령 제11조의2 및 별지 제8호 서식(훈련과정) \n한국기술교육대학교(직업능력심사평가원) 등의 심사결과에 따라 지방관서의 장이 인정·\n통지하는 훈련과정\n* \x07국민내일배움카드 사업내용 및 훈련과정 등 구체적인 내용은 고용24 홈페이지 (www.work24.\ngo.kr)에서 확인\n 취업정보 제공 \n •국방일보 취업정보 게재 (매주 화요일) : (http://kookbang.dema.mil.kr)\n •온라인(인터넷)을 통한 정보확인\n - \x07국방전직교육원(www.moti.or.kr) \n - 고용부 워크넷(www.wo

In [49]:
# 모든 testset의 데이터데 대한 llm 응답과 retriever의 검색 겷과를 추가
response_list = []
retrieved_context_list = []

for user_input in eval_df['user_input']:
    resp = chain.invoke(user_input)
    response_list.append(resp["response"])
    retrieved_context_list.append(resp["retrieved_context"])

In [38]:
print(len(response_list), len(retrieved_context_list))

8 8


In [39]:
response_list

['국방일보 취업정보는 **국방일보 홈페이지**에서 볼 수 있습니다.  \n경로는 다음과 같습니다.\n\n- **국방일보 취업정보 게재(매주 화요일)**: http://kookbang.dema.mil.kr\n\n',
 '국방일보의 취업정보는 **매주 화요일 국방일보에 게재**되며,  \n해당 정보는 **국방일보 홈페이지**에서 확인할 수 있습니다:\n\n- **http://kookbang.dema.mil.kr**\n\n또한 관련 취업정보는 온라인으로 다음 사이트에서도 확인 가능합니다:\n- 국방전직교육원: **www.moti.or.kr**\n- 고용부 워크넷: **www.work.go.kr**\n- 중소벤처기업진흥공단: **www.kosmes.or.kr**\n- 국가보훈부 제대군인지원센터: **www.vnet.go.kr**\n- 나라일터: **www.gojobs.go.kr**',
 '네, Context에 따르면 장교 선발계획은 **각 군 홈페이지**에 올라오며, 탑재 시기는 다음과 같습니다.\n\n- **육군**: 전반기 **전년도 12월**, 후반기 **8월**\n- **해군**: 전반기 **1월**\n- **공군**: 전반기 **2월**\n- **해병**: 전반기 **2월**, 후반기 **9월**\n\n또한 공군은 **선발결과만 홈페이지에 게시**되고, **계획은 공문으로 전파**된다고 되어 있습니다.\n\n장교로 장기복무를 준비할 때 꼭 확인해야 할 점은 다음과 같습니다.\n\n- **당해 연도 각 군별 장기복무 선발계획**\n- **선발절차**\n- **추진일정**\n- **선발방법 및 평가기준**\n- **매년 최신화되는 세부 평가요소**\n\n준비해야 할 사항으로는 다음이 안내되어 있습니다.\n\n- **본인 소관 업무를 책임감 있게 수행하기**\n- **상급자·하급자·동료와 원만한 관계 유지하기**\n- **해당 특기/분야에서 인정하는 자격증 등 자기개발 활동 하기**\n\n정보가 부족해 답을 할 수없습니다.',
 '부사관 장기복무 선발계획은 각 

In [40]:
retrieved_context_list

[['01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \n분\n야\n67 \n03\nPart\n2026 초급간부 길라잡이\n-③-Ⅵ\n국민내일배움카드(고용노동부) \n-③-Ⅶ\n기 타\n 개 요 \n \x07전 생애에 걸친 직무수행능력 습득·향상을 위해 국민 스스로 직업능력개발훈련을 \n실시할 수 있도록 훈련비 등 지원\n* 단, 관련 법령에 따라 공무원, 군인 등 일부 대상 제외\n* \x07군인의 경우, 국민 평생 직업능력 개발법 시행령 제6조 및 군인사법 시행령 제60조의2에 \n따라 단기복무 전역예정자의 경우 예외적으로 지원\n 주요내용\n •(신청자격) 신청일 기준 ① 5년 미만의 기간동안 현역으로 복무하고, \n ② 전역예정일까지의 기간이 6개월 이하인 장교·준사관 또는 부사관\n •(지원내용) 계좌발급일로부터 5년간 300만원 이내 훈련비 지원*\n* 훈련과정 직종별 취업률, 소득수준 등에 따라 훈련비 일정비율(0~55%) 자비 부담\n •(신청절차)\n신청인\n⇨\n각 군\n⇨\n관할 고용센터\n⇨\n훈련생·훈련기관\n• \x07신청서 작성 후 \n소속부대 담당자\n에게 제출\n•신청자격 확인 \n•\x07관할 고용센터에\n 추천 공문 발송\n•\x07신청인이 방문하여 \n발급절차 진행\n•\x07개인별 훈련과정 신청 \n및 실시\n* \x07(참고) 군 전직 및 취업 지원 업무에 관한 훈령 제11조의2 및 별지 제8호 서식(훈련과정) \n한국기술교육대학교(직업능력심사평가원) 등의 심사결과에 따라 지방관서의 장이 인정·\n통지하는 훈련과정\n* \x07국민내일배움카드 사업내용 및 훈련과정 등 구체적인 내용은 고용24 홈페이지 (www.work24.\ngo.kr)에서 확인\n 취업정보 제공 \n •국방일보 취업정보 게재 (매주 화요일) : (http://kookbang.dema.mil.kr)\n •온라인(인터넷)을 통한 정보확인\n - \x07국방전직교육원(www.moti.or.kr) \n - 고용부 워크넷(www.w

In [41]:
#
# eval_df에 컬럼으로 추가
#
eval_df['response'] = response_list
eval_df['retrieved_contexts'] = retrieved_context_list
eval_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,response,retrieved_contexts
0,국방일보 취업정보 어디서 볼수 있나요?,[01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \...,국방일보 취업정보는 매주 화요일 국방일보 홈페이지(http://kookbang.de...,Job Seeker in Career Transition,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer,국방일보 취업정보는 **국방일보 홈페이지**에서 볼 수 있습니다. \n경로는 다음...,[01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \...
1,국방일보에서 국민내일배움카드 관련 취업정보를 어떻게 확인할 수 있나요?,[01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \...,"국방일보에서는 매주 화요일 취업정보를 게재하며, 국민내일배움카드 관련 취업정보도 확...",Soon-to-be Discharged Service Member,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer,"국방일보의 취업정보는 **매주 화요일 국방일보에 게재**되며, \n해당 정보는 *...",[01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \...
2,장교 선발계획이 언제 어디에 올라오는지 알 수 있나요? 그리고 장교로 장기복무를 준...,[이 표는 **각 군(육군·해군·공군·해병)의 장기복무(연장복무) 선발 안내**를 ...,장교 선발계획은 각 군별로 홈페이지에 게시되는 시기가 다릅니다. 육군은 장교 전반기...,Soon-to-be Discharged Service Member,MISSPELLED,LONG,single_hop_specific_query_synthesizer,"네, Context에 따르면 장교 선발계획은 **각 군 홈페이지**에 올라오며, 탑...",[이 표는 **각 군(육군·해군·공군·해병)의 장기복무(연장복무) 선발 안내**를 ...
3,부사관 장기복무 선발계획은 언제 확인할 수 있습니까?,[이 표는 **각 군(육군·해군·공군·해병)의 장기복무(연장복무) 선발 안내**를 ...,"부사관 장기복무 선발계획은 육군 4월, 해군 전반기 1월과 후반기 6월, 공군 전반...",Military Career Progression Advisor,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,부사관 장기복무 선발계획은 각 군별로 홈페이지에 게시되는 시기가 다릅니다.\n\n-...,[01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n11 \n...
4,③ 취업알선 단계에서 제공되는 맞춤형 취업 지원 서비스에는 어떤 내용이 포함되어 있...,[이 표는 **취업알선 서비스의 구성 내용**을 정리한 것입니다. 주요 항목으로는 ...,"③ 취업알선 단계에서 제공되는 맞춤형 취업 지원 서비스에는 채용정보 제공, 직무 및...",Job Seeker in Career Transition,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer,"③ 취업알선 단계에서 제공되는 맞춤형 취업 지원 서비스에는 **채용정보 제공, 직무...",[이 표는 **취업알선 서비스의 구성 내용**을 정리한 것입니다. 주요 항목으로는 ...
5,취업알선 서비스에는 어떤 지원이 포함되어 있나요?,[이 표는 **취업알선 서비스의 구성 내용**을 정리한 것입니다. 주요 항목으로는 ...,"취업알선 서비스에는 채용정보 제공, 직무/기업분석, 그룹별 취업컨설팅, 희망기업 알...",Job Seeker in Career Transition,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer,"취업알선 서비스에는 **채용정보 제공, 직무/기업분석, 그룹별 취업컨설팅, 희망기업...","[이 표는 **취업알선 서비스**의 구성 내용을 정리한 것으로, 구직자에게 **채용..."
6,"국군재정관리단장 뭐 하는 사람인지 궁금해요, 사망보상금 받을 때 국군재정관리단장 역...",[Part 05 재해보상 분야\n58 2026 병 복지 길라잡이\n 재해보상금의 종...,사망보상금 받을 때 국군재정관리단장은 중요한 역할을 해요. 순직여부가 각 군 전공사...,Job Seeker in Career Transition,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer,"제공된 context에 따르면, **사망보상금 청구·지급 절차에서 국군재정관리단장*...",[01\nPart\n진\n료\n \n및\n \n재\n해\n보\n상\n \n분\n야\...
7,각 군 전공사상심사위원회는 어떤 역할을 하나요?,[Part 05 재해보상 분야\n58 2026 병 복지 길라잡이\n 재해보상금의 종...,각 군 전공사상심사위원회에서는 순직여부를 결정합니다.,Military Career Progression Advisor,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,[이 표는 **육군 간부 선발 또는 평가 시 항목별 배점 기준**을 정리한 것으로 ...


In [42]:
# eval_df를 ragas의 평가데이터셋 타입으로 변환
from ragas import EvaluationDataset
eval_dataset = EvaluationDataset.from_pandas(
    eval_df[["user_input", "retrieved_contexts", "response", "reference"]]
)
eval_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=8)

In [43]:
#
# 평가
#
from ragas.metrics import (
    LLMContextRecall,
    LLMContextPrecisionWithReference,
    Faithfulness,
    AnswerRelevancy,
)
from ragas import evaluate

C:\Users\Playdata\AppData\Local\Temp\ipykernel_37040\3378832119.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_37040\3378832119.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_37040\3378832119.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\User

In [44]:
# 평가할 때 사용할 LLM embedding 모델
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings)

]
eval_result = evaluate(dataset=eval_dataset, metrics=metrics)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_37040\384477035.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_37040\384477035.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))
Evaluating: 100%|██████████| 32/32 [03:08<00:00,  5.88s/it]


In [45]:
eval_result

{'context_recall': 0.8750, 'llm_context_precision_with_reference': 0.8299, 'faithfulness': 0.8393, 'answer_relevancy': 0.4647}

In [46]:
result_df = eval_result.to_pandas()
result_df

,user_input,retrieved_contexts,response,reference,context_recall,llm_context_precision_with_reference,faithfulness,answer_relevancy
0,국방일보 취업정보 어디서 볼수 있나요?,[01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \...,국방일보 취업정보는 **국방일보 홈페이지**에서 볼 수 있습니다. \n경로는 다음...,국방일보 취업정보는 매주 화요일 국방일보 홈페이지(http://kookbang.de...,1.0,1.000000,1.0000,0.582092
1,국방일보에서 국민내일배움카드 관련 취업정보를 어떻게 확인할 수 있나요?,[01\nPart\n자\n기\n개\n발\n \n및\n \n전\n직\n지\n원\n \...,"국방일보의 취업정보는 **매주 화요일 국방일보에 게재**되며, \n해당 정보는 *...","국방일보에서는 매주 화요일 취업정보를 게재하며, 국민내일배움카드 관련 취업정보도 확...",1.0,1.000000,1.0000,0.677288
2,장교 선발계획이 언제 어디에 올라오는지 알 수 있나요? 그리고 장교로 장기복무를 준...,[이 표는 **각 군(육군·해군·공군·해병)의 장기복무(연장복무) 선발 안내**를 ...,"네, Context에 따르면 장교 선발계획은 **각 군 홈페이지**에 올라오며, 탑...",장교 선발계획은 각 군별로 홈페이지에 게시되는 시기가 다릅니다. 육군은 장교 전반기...,1.0,1.000000,0.9375,0.000000
3,부사관 장기복무 선발계획은 언제 확인할 수 있습니까?,[01\nPart\n인\n사\n제\n도\n \n근\n무\n \n분\n야\n11 \n...,부사관 장기복무 선발계획은 각 군별로 홈페이지에 게시되는 시기가 다릅니다.\n\n-...,"부사관 장기복무 선발계획은 육군 4월, 해군 전반기 1월과 후반기 6월, 공군 전반...",1.0,0.638889,NaN,0.752006
4,③ 취업알선 단계에서 제공되는 맞춤형 취업 지원 서비스에는 어떤 내용이 포함되어 있...,[이 표는 **취업알선 서비스의 구성 내용**을 정리한 것입니다. 주요 항목으로는 ...,"③ 취업알선 단계에서 제공되는 맞춤형 취업 지원 서비스에는 **채용정보 제공, 직무...","③ 취업알선 단계에서 제공되는 맞춤형 취업 지원 서비스에는 채용정보 제공, 직무 및...",1.0,1.000000,1.0000,0.800423
5,취업알선 서비스에는 어떤 지원이 포함되어 있나요?,"[이 표는 **취업알선 서비스**의 구성 내용을 정리한 것으로, 구직자에게 **채용...","취업알선 서비스에는 **채용정보 제공, 직무/기업분석, 그룹별 취업컨설팅, 희망기업...","취업알선 서비스에는 채용정보 제공, 직무/기업분석, 그룹별 취업컨설팅, 희망기업 알...",1.0,1.000000,1.0000,0.905505
6,"국군재정관리단장 뭐 하는 사람인지 궁금해요, 사망보상금 받을 때 국군재정관리단장 역...",[01\nPart\n진\n료\n \n및\n \n재\n해\n보\n상\n \n분\n야\...,"제공된 context에 따르면, **사망보상금 청구·지급 절차에서 국군재정관리단장*...",사망보상금 받을 때 국군재정관리단장은 중요한 역할을 해요. 순직여부가 각 군 전공사...,1.0,1.000000,0.9375,0.000000
7,각 군 전공사상심사위원회는 어떤 역할을 하나요?,[이 표는 **육군 간부 선발 또는 평가 시 항목별 배점 기준**을 정리한 것으로 ...,정보가 부족해 답을 할 수없습니다.,각 군 전공사상심사위원회에서는 순직여부를 결정합니다.,0.0,0.000000,0.0000,0.000000
